# KRUMP_CORE_V1 — Fixed Benchmark

Base / epoch10 / epoch15 / final(epoch20) を完全固定条件で比較します。再学習・前処理・Dataset取得は行いません。


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, time, traceback, zipfile
WORK=Path('/content/KRUMP_BENCHMARK_OUTPUT')
RUNTIME=Path('/content/KRUMP_BENCHMARK_RUNTIME')
WORK.mkdir(parents=True,exist_ok=True); RUNTIME.mkdir(parents=True,exist_ok=True)
os.environ.update({'UV_CACHE_DIR':str(RUNTIME/'uv-cache'),'PIP_NO_CACHE_DIR':'1','HF_HOME':str(RUNTIME/'hf-cache'),'HUGGINGFACE_HUB_CACHE':str(RUNTIME/'hf-cache'/'hub'),'MPLBACKEND':'Agg'})
LOG=WORK/'benchmark.log'
def run(name,args,cwd=None,timeout=7200):
    p=subprocess.run(args,cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout,check=False)
    LOG.open('a',encoding='utf-8').write(f'\n===== {name} =====\n{p.stdout}\n'); print(p.stdout[-6000:])
    if p.returncode: raise RuntimeError(f'{name} failed (exit={p.returncode}); see {LOG}')
    return p.stdout


In [ ]:
import torch
print(subprocess.check_output(['nvidia-smi','-L'],text=True)); print('torch=',torch.__version__)
if not torch.cuda.is_available(): raise RuntimeError('CUDA unavailable')
GPU=torch.cuda.get_device_name(0); print('GPU=',GPU); print('arch=',torch.cuda.get_arch_list())
if not any(x in GPU for x in ('T4','L4','A100')): raise RuntimeError(f'Unsupported GPU: {GPU}; benchmark not started')


In [ ]:
from google.colab import files
uploaded=files.upload()
if set(uploaded) != {'KRUMP_CORE_V1_BENCHMARK_ADAPTERS.zip'}: raise RuntimeError('Upload exactly KRUMP_CORE_V1_BENCHMARK_ADAPTERS.zip')
adapter_zip=RUNTIME/'KRUMP_CORE_V1_BENCHMARK_ADAPTERS.zip'; adapter_zip.write_bytes(uploaded['KRUMP_CORE_V1_BENCHMARK_ADAPTERS.zip'])
ADAPTER_ROOT=RUNTIME/'adapters'
with zipfile.ZipFile(adapter_zip) as z: z.extractall(ADAPTER_ROOT)
ADAPTERS={'epoch10':ADAPTER_ROOT/'pilot_20'/'checkpoints'/'epoch_10_loss_0.5858','epoch15':ADAPTER_ROOT/'pilot_20'/'checkpoints'/'epoch_15_loss_0.6048','epoch20':ADAPTER_ROOT/'pilot_20'/'final'}
for name,path in ADAPTERS.items():
    if not (path/'adapter_config.json').exists() or not any(path.glob('*.safetensors')): raise RuntimeError(f'Invalid {name} adapter: {path}')
print('ADAPTER_INPUT_PASS', {k:str(v) for k,v in ADAPTERS.items()})


In [ ]:
ACE=RUNTIME/'ACE-Step-1.5'; CKPT=RUNTIME/'checkpoints'
run('install uv',['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],timeout=300)
UV=shutil.which('uv') or '/root/.local/bin/uv'
if not ACE.exists(): run('clone',['git','clone','https://github.com/ace-step/ACE-Step-1.5.git',str(ACE)],timeout=900)
run('pin',['git','checkout','7202bc354d7fc31d1c0e5a90b0b49fb610e52362'],cwd=ACE,timeout=120)
run('deps',[UV,'sync','--no-cache'],cwd=ACE,timeout=2400); CKPT.mkdir(exist_ok=True)
link=ACE/'checkpoints'
if link.exists() or link.is_symlink(): link.unlink() if (link.is_symlink() or link.is_file()) else shutil.rmtree(link)
link.symlink_to(CKPT,target_is_directory=True)
run('base model',[UV,'run','--no-sync','acestep-download','--model','acestep-v15-base','--dir',str(CKPT)],cwd=ACE,timeout=3600)
if not any(CKPT.rglob('*.safetensors')): raise RuntimeError('Base weights missing')
print('ACE_STEP_BASE_READY')


In [ ]:
benchmark_script=RUNTIME/'run_benchmark.py'
benchmark_script.write_text(r'''from pathlib import Path
import json, os, zipfile, gc, time
import torch
import soundfile as sf
WORK=Path('/content/KRUMP_BENCHMARK_OUTPUT'); RUNTIME=Path('/content/KRUMP_BENCHMARK_RUNTIME'); ACE=RUNTIME/'ACE-Step-1.5'; CKPT=RUNTIME/'checkpoints'; OUT=WORK/'fixed_120_csharp_minor'; OUT.mkdir(parents=True,exist_ok=True)
PROMPT='instrumental KRUMP, dark raw KRUMP, strong kick, strong snare, unstable dissonant melody, distinctive rhythmic spacing and accents, minimal aggressive 32-bar loop structure, avoid generic trap or rage'
SETTINGS={'caption':PROMPT,'bpm':120,'key_scale':'C# minor','time_signature':'4','duration_seconds':64.0,'bars_equivalent':32,'instrumental':True,'lyrics':'[Instrumental]','seed':20260831,'inference_steps':32,'guidance_scale':7.0,'use_adg':True,'shift':3.0,'infer_method':'ode','batch_size':1,'audio_format':'wav','model_variant':'acestep-v15-base'}
ADAPTERS={'epoch10':RUNTIME/'adapters'/'pilot_20'/'checkpoints'/'epoch_10_loss_0.5858','epoch15':RUNTIME/'adapters'/'pilot_20'/'checkpoints'/'epoch_15_loss_0.6048','epoch20':RUNTIME/'adapters'/'pilot_20'/'final'}
os.environ['ACESTEP_CHECKPOINTS_DIR']=str(CKPT); os.environ['ACESTEP_DISABLE_TQDM']='1'
from acestep.handler import AceStepHandler
def save(result,dst):
    if not result.get('success') or not result.get('audios'): raise RuntimeError(result.get('error') or result.get('status_message') or 'No audio returned')
    item=result['audios'][0]; audio=item['tensor'].detach().float().cpu(); data=audio.numpy() if audio.ndim==1 else audio.transpose(0,1).numpy(); sf.write(str(dst),data,item['sample_rate'],subtype='PCM_16')
    if not dst.exists() or dst.stat().st_size<4096: raise RuntimeError(f'WAV missing: {dst}')
    return {'path':str(dst),'bytes':dst.stat().st_size,'sample_rate':item['sample_rate']}
def generate(handler,dst):
    return save(handler.generate_music(captions=SETTINGS['caption'],lyrics=SETTINGS['lyrics'],bpm=SETTINGS['bpm'],key_scale=SETTINGS['key_scale'],time_signature=SETTINGS['time_signature'],vocal_language='unknown',inference_steps=SETTINGS['inference_steps'],guidance_scale=SETTINGS['guidance_scale'],use_random_seed=False,seed=SETTINGS['seed'],audio_duration=SETTINGS['duration_seconds'],batch_size=SETTINGS['batch_size'],task_type='text2music',use_adg=SETTINGS['use_adg'],shift=SETTINGS['shift'],infer_method=SETTINGS['infer_method']),dst)
started=time.time(); handler=AceStepHandler(); status,ok=handler.initialize_service(project_root=str(ACE),config_path='acestep-v15-base',device='cuda',use_flash_attention=False,compile_model=False,offload_to_cpu=False,offload_dit_to_cpu=False,quantization=None,use_mlx_dit=False)
if not ok: raise RuntimeError('Base model initialization failed: '+status)
report={'settings':SETTINGS,'base_init':status,'outputs':{},'adapters':{k:str(v) for k,v in ADAPTERS.items()}}
report['outputs']['base']=generate(handler,OUT/'base.wav')
for label in ('epoch10','epoch15','epoch20'):
    loaded=handler.load_lora(str(ADAPTERS[label]))
    if not loaded.startswith('✅'): raise RuntimeError(f'Failed to load {label}: {loaded}')
    report['outputs'][label]=generate(handler,OUT/f'{label}.wav')
    unloaded=handler.unload_lora()
    if not unloaded.startswith('✅'): raise RuntimeError(f'Failed to unload {label}: {unloaded}')
    gc.collect(); torch.cuda.empty_cache()
cfg=OUT/'generation_config.json'; report['elapsed_seconds']=round(time.time()-started,2); cfg.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
required=[OUT/'base.wav',OUT/'epoch10.wav',OUT/'epoch15.wav',OUT/'epoch20.wav',cfg]
archive=Path('/content/KRUMP_CORE_V1_FIXED_BENCHMARK.zip')
if archive.exists(): archive.unlink()
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in required:
        if not p.exists(): raise RuntimeError(f'Required output missing: {p}')
        z.write(p,arcname=p.name)
with zipfile.ZipFile(archive) as z:
    if sorted(z.namelist()) != sorted(p.name for p in required): raise RuntimeError(f'Unexpected ZIP contents: {z.namelist()}')
report['zip']=str(archive); report['zip_bytes']=archive.stat().st_size; cfg.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps({'status':'BENCHMARK_PASS','outputs':report['outputs'],'zip':str(archive),'zip_bytes':archive.stat().st_size,'elapsed_seconds':report['elapsed_seconds']},indent=2))
''',encoding='utf-8')
p=subprocess.run([UV,'run','--no-sync','python',str(benchmark_script)],cwd=ACE,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=21600,check=False)
(WORK/'benchmark_runner.log').write_text(p.stdout,encoding='utf-8'); print(p.stdout[-12000:])
if p.returncode: raise RuntimeError(f'Benchmark failed (exit={p.returncode}); see {WORK}/benchmark_runner.log')
from google.colab import files
files.download('/content/KRUMP_CORE_V1_FIXED_BENCHMARK.zip')
